In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
Weather_API_Key = os.getenv("WEATHER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

from langchain.chat_models import init_chat_model

llm = init_chat_model(model="openai/gpt-oss-120B", model_provider="groq")

In [2]:
from langchain.tools import tool
import requests
url = "https://api.openweathermap.org/data/2.5/weather?"
@tool
def get_weather(location:str)->str:
    """Take the location and return the current weather. """
    params={
        "q": location, 
        "appid" : Weather_API_Key, 
        "units" : "imperial"
    }

    try: 
        response = requests.get(url, params=params, timeout = 10)
        return response.json()
    except:
        return f"The requested weather information is unavailable."

In [3]:
from langchain_tavily import TavilySearch
web_search = TavilySearch(
    max_results=3, 
    topic="general", 
    search_depth="basic"
)

In [4]:
llms_with_tools = llm.bind_tools([get_weather, web_search])

In [5]:
from langchain.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(
        content=(
            "You are a helpful agent. Use the getweather tool for weather information, "
            "web_search for internet search/news, or both if a query requires multiple sources."
        )
    )
]

In [6]:
user_input = input("What is your query? ")
messages.append(HumanMessage(content=user_input))

tools_map = {get_weather.name: get_weather, web_search.name: web_search}
executed_tools = []

while True:
    aiMessage = llms_with_tools.invoke(messages)
    messages.append(aiMessage)
    
    if not aiMessage.tool_calls:
        final_response = aiMessage
        break
        
    for tool_call in aiMessage.tool_calls:
        tool_name = tool_call["name"]
        if tool_name in tools_map:
            print(f"Executing tool: {tool_name} with args: {tool_call['args']}")
            tool_message = tools_map[tool_name].invoke(tool_call)
            messages.append(tool_message)
            executed_tools.append(tool_name)

Executing tool: tavily_search with args: {'query': 'who won the 2026 FIFA World Cup', 'search_depth': 'fast', 'time_range': 'year'}
Executing tool: tavily_search with args: {'query': '2026 FIFA World Cup winner', 'search_depth': 'advanced', 'time_range': 'year', 'topic': 'news'}
Executing tool: get_weather with args: {'location': 'Peoria, IL'}
Executing tool: get_weather with args: {'location': 'Peoria, Illinois, USA'}


In [7]:
print("\nExecuted Tool(s):", ", ".join(executed_tools) if executed_tools else "None")
print("Final response:\n", final_response.content)


Executed Tool(s): tavily_search, tavily_search, get_weather, get_weather
Final response:
 **2026 FIFA World Cup (Men’s) – Champion**

- **Winner:** **Spain**  
- **Final result:** Spain defeated Argentina 1–0 after extra‑time on 19 July 2026 to claim the title【2†results.0.content】.  

**Current weather in Peoria, Illinois (USA)**  

- **Condition:** Clear sky (very few clouds)  
- **Temperature:** **84.8 °F** (feels like 86.7 °F)  
- **Humidity:** 52 %  
- **Wind:** 2 m/s (≈ 4.5 mph) from the southwest, gusts up to 7.6 m/s  
- **Sunrise / Sunset:** Sunrise at 06:44 local, Sunset at 19:31 local (times shown in the API data)  

*Source for weather:* OpenWeather data for “Peoria, Illinois, USA” returned on request【3†】.
